In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# @title Libraries
from typing import Mapping
import os
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import librosa
import matplotlib.pyplot as plt
from torch import optim
import time
import random
import matplotlib.axes as ax
import pandas as pd
import seaborn as sn

In [36]:
# @title Constants
class_to_int = {
    'neutral' : 0,
    'calm' : 1,
    'happy' : 2,
    'sad': 3,
    'angry': 4,
    'fearful': 5,
    'disgust': 6,
    'surprised': 7
}

int_to_class = {v: k for k, v in class_to_int.items()}

data_pth = '/content/drive/MyDrive/COSMOSPosterProject/RAVDESS_DATASET/combined'
arr_pth = '/content/drive/MyDrive/COSMOSPosterProject/EmoNet/data_arrs'

SAMPLE_RATE = 48000

train_arrs = os.path.join(arr_pth, 'train_arrs')
test_arrs = os.path.join(arr_pth, 'test_arrs_updated')

In [7]:
# @title Custom Dataset
def add_noise(signal, num_bits=16, augmented_num=2, snr=30):
    signal_len = len(signal)
    # Generate White Gaussian noise
    noise = np.random.normal(size=(augmented_num, signal_len))
    # Normalize signal and noise
    norm_constant = 2.0**(num_bits-1)
    signal_norm = signal / norm_constant
    noise_norm = noise / norm_constant
    # Compute signal and noise power
    s_power = np.sum(signal_norm ** 2) / signal_len
    n_power = np.sum(noise_norm ** 2, axis=1) / signal_len
    # Random SNR: Uniform [15, 30] in dB
    target_snr = os.ST_NODIRATIME
    # Compute K (covariance matrix) for each noise
    K = np.sqrt((s_power / n_power) * 10 ** (- target_snr / 10))
    K = np.ones((signal_len, augmented_num)) * K
    # Generate noisy signal
    return signal + K.T * noise

def add_noise_image(image, mean=0, var=0.01):
    image_arr = np.array(image)
    sigma = np.sqrt(var)

    # Generate Gaussian noise
    gauss_noise = np.random.normal(mean, sigma, image_arr.shape)

    # Add noise to the image
    noisy_image_array = image_arr + gauss_noise

    # Clip the values to be in valid range
    noisy_image_array = np.clip(noisy_image_array, 0, 255)

    noisy_image = Image.fromarray(np.uint8(noisy_image_array))

    return noisy_image

# Define the transforms for grayscale images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

class CombinedDataset(Dataset):
    def __init__(self, root_dir, transform=transform, max_frames=35, offset=0, add_noise=True, noise_idx=0, sound_snr=30, image_var=0):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.class_to_idx = class_to_int
        self.video_files = []
        self.audio_files = []
        self.labels = []
        self.max_frames = max_frames
        self.offset = offset
        self.add_noise = add_noise
        self.noise_idx = noise_idx
        self.sound_snr = sound_snr
        self.image_var = image_var

        for class_name in self.classes:
            class_dir = os.path.join(self.root_dir, class_name)
            if os.path.isdir(class_dir):
                for mini_dir in os.listdir(class_dir):
                    mini_path = os.path.join(class_dir, mini_dir)
                    files = os.listdir(mini_path)
                    if files[0].endswith('.mp4'):
                        self.video_files.append(os.path.join(mini_path, files[0]))
                        self.audio_files.append(os.path.join(mini_path, files[1]))
                        self.labels.append(self.class_to_idx[class_name])
                    else:
                        self.video_files.append(os.path.join(mini_path, files[1]))
                        self.audio_files.append(os.path.join(mini_path, files[0]))
                        self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        #print(f"Worker processing index {idx}")
        video_path = self.video_files[idx]
        audio_path = self.audio_files[idx]
        label = self.labels[idx]
        #print(f"Worker {idx} beginning frames")
        frames = self.get_vid(video_path)
        #print(f"Worker {idx} getting audio")
        spec = self.get_spec(audio_path)
        #print(f"Worker {idx} finished!")
        return frames, spec, label

    def print_files(self):
        for i in range(len(self.video_files)):
            print('Video file: ' + self.video_files[i].split('/')[-1] + ' Audio file: ' + self.audio_files[i].split('/')[-1] + ' label: ' + str(self.labels[i]))

    def get_spec(self, path):
        t = time.time()
        audio, sample_rate = librosa.load(path, duration=3, sr=SAMPLE_RATE)
        signal = np.zeros((int(SAMPLE_RATE*3, )))
        signal[:len(audio)] = audio
        if (self.add_noise):
            signal = add_noise(signal)[self.noise_idx]
        mel_spec = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_fft=1024, win_length = 512, window='hamming', hop_length = 256, n_mels=128, fmax=sample_rate/2)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        """
        print(mel_spec_db.shape)
        plt.figure(figsize=(12, 4))
        librosa.display.specshow(mel_spec_db, sr=SAMPLE_RATE, x_axis='time', y_axis='hz')
        plt.colorbar(format='%+2.0f dB')
        plt.title('Spectrogram')
        plt.show()
        """
        #print("Time Audio: " + str(time.time() - t))
        return torch.tensor(mel_spec_db)


    def get_vid(self, video_path):
        cap = cv2.VideoCapture(video_path)
        frames = []
        t = time.time()
        for i in range(self.max_frames):
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to grayscale
            RGB_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            faces = face_cascade.detectMultiScale(RGB_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
            if len(faces) > 0:
                # Sort faces by size (largest face first)
                faces = sorted(faces, key=lambda x: x[2] * x[3], reverse=True)

                # Take the first detected face (largest face)
                (x, y, w, h) = faces[0]

                # Crop the face region from the frame
                face = frame[y:y+h, x:x+w]

                # Convert to PIL Image for transformation
                pil_face = Image.fromarray(face)

                if self.transform:
                    face_tensor = self.transform(pil_face)
                    face_tensor = torch.squeeze(face_tensor)  # Remove single-dimensional entries
                    frames.append(face_tensor)
        cap.release()

        frames = torch.stack(frames)
        frames = frames.permute(1, 0, 2, 3)
        return frames

# @title Datasets (Not Used ATM)
train_dataset = ConcatDataset([CombinedDataset(os.path.join(data_pth, 'train'),add_noise=False),
                               CombinedDataset(os.path.join(data_pth, 'train'),add_noise=True, sound_snr=5),
                               CombinedDataset(os.path.join(data_pth, 'train'),add_noise=True, sound_snr=10)])
test_dataset = ConcatDataset([CombinedDataset(os.path.join(data_pth, 'test'),add_noise=False),
                              CombinedDataset(os.path.join(data_pth, 'test'),add_noise=True, sound_snr=5),
                              CombinedDataset(os.path.join(data_pth, 'test'),add_noise=True, sound_snr=10)])


In [ ]:
test_arr = []
ls = [[], [], []]
for i in range(len(test_dataset)):
    test = test_dataset[i]
    ls[0].append(test[0])
    ls[1].append(test[1])
    ls[2].append(test[2])
    print('counter: ' + str(i))
    if (len(ls[0]) == 32):
        test_arr.append(ls)
        ls = [[], [], []]

counter: 0
counter: 1
counter: 2
counter: 3
counter: 4
counter: 5
counter: 6
counter: 7
counter: 8
counter: 9
counter: 10
counter: 11
counter: 12
counter: 13
counter: 14
counter: 15
counter: 16
counter: 17
counter: 18
counter: 19
counter: 20
counter: 21
counter: 22
counter: 23
counter: 24
counter: 25
counter: 26
counter: 27
counter: 28
counter: 29
counter: 30
counter: 31
counter: 32
counter: 33
counter: 34
counter: 35
counter: 36
counter: 37
counter: 38
counter: 39
counter: 40
counter: 41
counter: 42
counter: 43
counter: 44
counter: 45
counter: 46
counter: 47
counter: 48
counter: 49
counter: 50
counter: 51
counter: 52
counter: 53
counter: 54
counter: 55
counter: 56
counter: 57
counter: 58
counter: 59
counter: 60
counter: 61
counter: 62
counter: 63
counter: 64
counter: 65
counter: 66
counter: 67
counter: 68
counter: 69
counter: 70
counter: 71
counter: 72
counter: 73
counter: 74
counter: 75
counter: 76
counter: 77
counter: 78
counter: 79
counter: 80
counter: 81
counter: 82
counter: 83
co

In [32]:
for i in range(32767):
    idx1 = random.randint(0, 1855)
    idx2 = random.randint(0, 1855)
    for j in range(len(ls)):
        ls[j][idx1], ls[j][idx2] = ls[j][idx2], ls[j][idx1]

In [44]:
ls = [[], [], []]
shp = test_arr[0][0][0].shape
for i in range(len(test_arr)):
    for j in range(len(test_arr[i][0])):
        print(test_arr[i][0][j].shape)
        if (test_arr[i][0][j].shape == shp):
            for k in range(len(test_arr[i])):
                ls[k].append(test_arr[i][k][j])

torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size

In [48]:
ls_batch = []
curr_arr = [[], [], []]
for i in range(len(ls[0])):
    curr_arr[0].append(ls[0][i])
    curr_arr[1].append(ls[1][i])
    curr_arr[2].append(ls[2][i])
    if (len(curr_arr[0]) == 32):
        ls_batch.append(curr_arr)
        curr_arr = [[], [], []]

In [49]:
print(len(ls_batch))

57


In [52]:
for i in range(len(ls_batch)):
    video = torch.stack(ls_batch[i][0])
    #print(video.shape)
    audio = torch.stack(ls_batch[i][1])
    #print(audio.shape)
    label = torch.tensor(ls_batch[i][2])
    #print(label.shape)
    torch.save([video, audio, label], os.path.join(test_arrs, f'batch{i}.pt'))

IndexError: list index out of range

In [34]:
global_cntr = {
    'neutral':0,
    'calm':0,
    'happy':0,
    'sad':0,
    'angry':0,
    'fearful':0,
    'disgust':0,
    'surprised':0
}
for i, batch in enumerate(test_arr):
    cntr = {
        'neutral':0,
        'calm':0,
        'happy':0,
        'sad':0,
        'angry':0,
        'fearful':0,
        'disgust':0,
        'surprised':0
    }
    labels = batch[2]
    for label in labels:
        cntr[int_to_class[label]] += 1
        global_cntr[int_to_class[label]] += 1
    print(f'Batch {i+1}: {cntr}')

Batch 1: {'neutral': 32, 'calm': 0, 'happy': 0, 'sad': 0, 'angry': 0, 'fearful': 0, 'disgust': 0, 'surprised': 0}
Batch 2: {'neutral': 16, 'calm': 16, 'happy': 0, 'sad': 0, 'angry': 0, 'fearful': 0, 'disgust': 0, 'surprised': 0}
Batch 3: {'neutral': 0, 'calm': 32, 'happy': 0, 'sad': 0, 'angry': 0, 'fearful': 0, 'disgust': 0, 'surprised': 0}
Batch 4: {'neutral': 0, 'calm': 32, 'happy': 0, 'sad': 0, 'angry': 0, 'fearful': 0, 'disgust': 0, 'surprised': 0}
Batch 5: {'neutral': 0, 'calm': 16, 'happy': 16, 'sad': 0, 'angry': 0, 'fearful': 0, 'disgust': 0, 'surprised': 0}
Batch 6: {'neutral': 0, 'calm': 0, 'happy': 32, 'sad': 0, 'angry': 0, 'fearful': 0, 'disgust': 0, 'surprised': 0}
Batch 7: {'neutral': 0, 'calm': 0, 'happy': 32, 'sad': 0, 'angry': 0, 'fearful': 0, 'disgust': 0, 'surprised': 0}
Batch 8: {'neutral': 0, 'calm': 0, 'happy': 16, 'sad': 16, 'angry': 0, 'fearful': 0, 'disgust': 0, 'surprised': 0}
Batch 9: {'neutral': 0, 'calm': 0, 'happy': 0, 'sad': 32, 'angry': 0, 'fearful': 0, '